In [1]:
# =============================================================================
# Cell 1: Imports and Spark Session
# =============================================================================
import sys
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import DoubleType
import math

# Import BERDL Spark session utility
# Configure Spark session
sys.path.append('/opt/conda/lib/python3.13/site-packages')
from berdl_notebook_utils.setup_spark_session import get_spark_session

spark = get_spark_session()
sns.set_theme(style="whitegrid", palette="muted")
print("Spark session acquired.")

> **PENDING** — The db-RDA result (R²=0.799) from this notebook is **invalidated**: the original implementation used random `project_accession` labels and a circular predictor derived from the COG PCA itself. Do not cite. To rerun correctly, add `F.col("s.project_accession")` to the SELECT in Cell 3, increase `sample_limit` in Cell 8 from 500 to ≥2000, and rerun Cells 3, 8, and 9.

In [2]:
# =============================================================================
# Cell 2: Haversine UDF for Spatial Distance
# =============================================================================
def haversine(lat1, lon1, lat2, lon2):
    if None in (lat1, lon1, lat2, lon2):
        return None
    R = 6371.0
    lat1_rad = math.radians(lat1)
    lat2_rad = math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

haversine_udf = F.udf(haversine, DoubleType())

In [3]:
# =============================================================================
# Cell 3: Load Soil Metal Data from arkinlab_microbeatlas
# =============================================================================
SOIL_DB = "arkinlab_microbeatlas"
KBASE_DB = "kbase_ke_pangenome"

df_sample = spark.table(f"{SOIL_DB}.sample_metadata")
df_enriched = spark.table(f"{SOIL_DB}.enriched_metadata")
df_gee = spark.table(f"{SOIL_DB}.enriched_metadata_gee")

# Deduplicate
df_enriched = df_enriched.withColumn("rn", F.row_number().over(Window.partitionBy("accession_id").orderBy("accession_id"))).filter(F.col("rn") == 1).drop("rn")
df_gee = df_gee.withColumn("rn", F.row_number().over(Window.partitionBy("SRS_Join_Key").orderBy("SRS_Join_Key"))).filter(F.col("rn") == 1).drop("rn")

# Join
df_soil_raw = (
    df_sample.alias("s")
    .join(df_enriched.alias("e"), F.col("s.sample_id") == F.col("e.accession_id"), "inner")
    .join(df_gee.alias("g"), F.col("s.SRS_Join_Key") == F.col("g.SRS_Join_Key"), "inner")
)

# Filter soil samples with OTU data and coordinates
df_soil_raw = df_soil_raw.filter(
    (F.col("s.SRS_Join_Key").isNotNull()) &
    (F.col("s.n_genes_by_counts").isNotNull()) &
    (F.col("s.Env_Level_1") == "soil")
).filter(
    F.col("g.LatitudeParsed").isNotNull() & F.col("g.LongitudeParsed").isNotNull()
)

# Metal columns
metal_columns = [
    "e.GeoROC_Rocks_georoc_Co_ppm", "e.GeoROC_Rocks_georoc_Cr_ppm",
    "e.GeoROC_Rocks_georoc_Cu_ppm", "e.GeoROC_Rocks_georoc_Ni_ppm",
    "e.GeoROC_Rocks_georoc_Zn_ppm", "e.GeoROC_Rocks_georoc_Pb_ppm",
    "e.GeoROC_Rocks_georoc_As_ppm", "e.GeoROC_Rocks_georoc_Cd_ppm",
    "e.GeoROC_Rocks_georoc_Hg_ppm"
]

df_soil_metal = df_soil_raw.select(
    F.col("s.sample_id"),
    F.col("g.Project").alias("project_accession"),
    F.col("g.LatitudeParsed").alias("lat"),
    F.col("g.LongitudeParsed").alias("lon"),
    F.col("g.ph").alias("ph"),
    *[F.col(c).cast("double").alias(c.split(".")[-1]) for c in metal_columns]
).cache()

print(f"Soil samples with metal data: {df_soil_metal.count()}")
df_soil_metal.show(5, truncate=False)

In [4]:
# =============================================================================
# Cell 4: Load KBase Genome Data with COG Annotations and Coordinates
# =============================================================================
# Coordinates
df_kbase_coords = spark.table(f"{KBASE_DB}.alphaearth_embeddings_all_years") \
    .select(
        F.col("genome_id"),
        F.expr("TRY_CAST(cleaned_lat AS DOUBLE)").alias("lat"),
        F.expr("TRY_CAST(cleaned_lon AS DOUBLE)").alias("lon")
    ).filter(F.col("lat").isNotNull())

# eggNOG annotations
df_eggnog = spark.table(f"{KBASE_DB}.eggnog_mapper_annotations") \
    .select(
        F.col("query_name").alias("gene_id"),
        F.col("COG_category")
    )

# Gene to genome mapping
df_gene_map = spark.table(f"{KBASE_DB}.gene") \
    .select("gene_id", "genome_id")

# Count COG categories per genome (long format)
df_genome_cog_long = df_gene_map.join(df_eggnog, "gene_id", "inner") \
    .groupBy("genome_id", "COG_category").count()

# Pivot to wide format
df_genome_cog = df_genome_cog_long.groupBy("genome_id").pivot("COG_category").sum("count").fillna(0)

# Join coordinates with COG counts
df_kbase_cog = df_kbase_coords.join(df_genome_cog, "genome_id", "left").fillna(0)

print(f"KBase genomes with COG and coordinates: {df_kbase_cog.count()}")
df_kbase_cog.show(5, truncate=False)

In [5]:
# =============================================================================
# Cell 5: Spatial Join (10 km radius)
# =============================================================================
RADIUS_KM = 10.0

# Binned coordinates
spark_soil_binned = df_soil_metal.withColumn("lat_bin", F.round("lat", 1)).withColumn("lon_bin", F.round("lon", 1))
spark_kbase_binned = df_kbase_cog.withColumn("lat_bin", F.round("lat", 1)).withColumn("lon_bin", F.round("lon", 1))

# Join on bins
joined = spark_soil_binned.alias("soil").join(
    spark_kbase_binned.alias("kb"),
    on=["lat_bin", "lon_bin"],
    how="inner"
)

# Filter by exact distance
nearby = joined.withColumn(
    "distance_km",
    haversine_udf(F.col("soil.lat"), F.col("soil.lon"), F.col("kb.lat"), F.col("kb.lon"))
).filter(F.col("distance_km") <= RADIUS_KM).cache()

total_soil = df_soil_metal.count()
distinct_soil = nearby.select("soil.sample_id").distinct().count()
print(f"Soil samples with ≥1 genome within {RADIUS_KM} km: {distinct_soil} (out of {total_soil})")

In [6]:
# =============================================================================
# Cell 6: Aggregate and Normalize COG Counts per Soil Sample (Spatial)
# =============================================================================
cog_columns = [c for c in df_genome_cog.columns if c != "genome_id"]

# Count nearby genomes per sample
df_genome_count = nearby.groupBy("soil.sample_id").agg(F.count("kb.genome_id").alias("num_nearby_genomes"))

# Sum COG counts
agg_exprs = [F.sum(F.col(f"kb.{c}")).alias(f"sum_{c}") for c in cog_columns]
df_cog_sum = nearby.groupBy("soil.sample_id").agg(*agg_exprs)

# Normalize: average COG count per genome
norm_exprs = [(F.col(f"sum_{c}") / F.col("num_nearby_genomes")).alias(f"avg_{c}") for c in cog_columns]
df_cog_norm = df_cog_sum.join(df_genome_count, "sample_id").select("sample_id", "num_nearby_genomes", *norm_exprs)

# Join back to soil metal
df_metal_cog = df_soil_metal.join(df_cog_norm, "sample_id", "left").fillna(0)

# Convert to Pandas (downsample for performance)
pdf_metal_cog = df_metal_cog.sample(fraction=0.3, seed=42).toPandas()
print(f"Spatial COG dataset (downsampled): {pdf_metal_cog.shape}")

In [7]:
# =============================================================================
# Cell 7: Load GTDB Taxonomy and OTU Order Abundance
# =============================================================================
# GTDB taxonomy for KBase genomes
df_gtdb = spark.table(f"{KBASE_DB}.gtdb_taxonomy_r214v1") \
    .select("genome_id", "domain", "phylum", "class", "order", "family", "genus", "species")

# OTU counts and taxonomy
df_otu = spark.table(f"{SOIL_DB}.otu_counts_long")
df_otu_tax = spark.table(f"{SOIL_DB}.otu_metadata")

# Parse taxonomy string into ranks
ranks = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
df_otu_tax = df_otu_tax.withColumn("tax_array", F.split(F.col("Tax"), ";"))
for i, rank in enumerate(ranks):
    df_otu_tax = df_otu_tax.withColumn(rank, F.try_element_at(F.col("tax_array"), F.lit(i+1)))
df_otu_tax = df_otu_tax.drop("tax_array")

# Join OTU counts with Order
df_otu_order = df_otu.join(df_otu_tax.select("otu_id", "Order"), "otu_id", "inner") \
    .groupBy("sample_id", "Order").agg(F.sum("count").alias("abundance"))

# Relative abundance per Order per sample
sample_totals = df_otu_order.groupBy("sample_id").agg(F.sum("abundance").alias("total"))
df_order_rel = df_otu_order.join(sample_totals, "sample_id") \
    .withColumn("rel_abundance", F.col("abundance") / F.col("total")) \
    .select("sample_id", "Order", "rel_abundance")

print(f"Order-level relative abundance: {df_order_rel.count()} rows")

In [8]:
# =============================================================================
# Cell 8: Ultra-optimized Taxonomic Join (Demonstration-ready)
# =============================================================================
from pyspark.sql.functions import broadcast
def clean_taxon(col):
    return F.lower(F.regexp_replace(F.col(col), "^[a-z]__", ""))

# Clean Order names
df_order_rel_clean = df_order_rel.withColumn("order_clean", clean_taxon("Order"))
df_gtdb_clean = df_gtdb.withColumn("order_clean", clean_taxon("order"))

# Join COG with GTDB order
df_genome_cog_order = df_genome_cog.join(df_gtdb_clean, "genome_id", "inner")

# --- DRASTIC OPTIMIZATION ---
# 1. Take only a small random subset of soil samples (e.g., 500 samples)
sample_limit = 5000
soil_sample_ids = df_order_rel_clean.select("sample_id").distinct().limit(sample_limit)
df_order_rel_subset = df_order_rel_clean.join(soil_sample_ids, "sample_id", "inner")

# 2. Keep only the top 20 most abundant orders per sample (using window)
from pyspark.sql.window import Window
window_spec = Window.partitionBy("sample_id").orderBy(F.desc("rel_abundance"))
df_order_rel_top = df_order_rel_subset.withColumn("rank", F.row_number().over(window_spec)).filter(F.col("rank") <= 20).drop("rank")

print(f"Subset soil rows: {df_order_rel_top.count()}")

# 3. Pre-aggregate genome side: average COG counts per order
genome_order_agg = df_genome_cog_order.groupBy("order_clean").agg(
    *[F.avg(F.col(c)).alias(f"avg_{c}") for c in cog_columns]
)

# 4. Join the small soil subset with aggregated genome table
joined_tax = df_order_rel_top.alias("soil").join(
    broadcast(genome_order_agg.alias("kb")),
    on="order_clean",
    how="inner"
)

# Weighted sum using pre-averaged COG values
weighted_exprs = [F.sum(F.col("soil.rel_abundance") * F.col(f"kb.avg_{c}")).alias(f"weighted_{c}") for c in cog_columns]
df_community_cog = joined_tax.groupBy("soil.sample_id").agg(*weighted_exprs)

# Join with soil metal
df_metal_community = df_soil_metal.join(df_community_cog, "sample_id", "left").fillna(0)

# Convert to Pandas
pdf_community = df_metal_community.toPandas()
print(f"Community-weighted COG dataset shape: {pdf_community.shape}")

In [9]:
# =============================================================================
# Cell 9 (Revised): Unconditional db-RDA — Metals → Community Functional Profiles
# =============================================================================
# Uses Aitchison distance (CLR transform + PCA) on COG profiles.
# Unconditional R²: variance in COG profiles explained by metals alone.
# Conditional R² (metals | project_accession): add project_accession to the SELECT
#   in cell 3 and rerun cells 3, 8, and this cell.
#
# NOTE: the prior implementation used random project IDs (np.random.choice) and a
# circular predictor derived from the COG PCA itself. R²=0.799 from that run is an
# artifact and should not be cited.

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# --- 1. Filter to samples with actual community COG data and ≥3 metals ---
cog_weighted_cols = [c for c in pdf_community.columns if c.startswith('weighted_')]
metal_cols        = [c for c in pdf_community.columns if 'georoc' in c and c.endswith('_ppm')]

has_cog   = pdf_community[cog_weighted_cols].sum(axis=1) > 0
has_metal = (~pdf_community[metal_cols].isna()).sum(axis=1) >= 3
pdf_rda   = pdf_community[has_cog & has_metal].reset_index(drop=True)
print(f"Samples with community COG data + ≥3 metals: {len(pdf_rda)} (of {len(pdf_community)})")
print(f"Metal predictors ({len(metal_cols)}): {metal_cols}")

if len(pdf_rda) < 20:
    raise RuntimeError(
        f"Only {len(pdf_rda)} usable samples. "
        "Increase sample_limit in cell 8 (community join) then rerun cells 8–9."
    )

# --- 2. COG response matrix: CLR transform → PCA (≡ PCoA on Aitchison distance) ---
X_cog  = pdf_rda[cog_weighted_cols].values.astype(float)
X_cog  = np.where(X_cog <= 0, 1e-9, X_cog)
X_clr  = np.log(X_cog) - np.log(X_cog).mean(axis=1, keepdims=True)
X_clr_sc = StandardScaler().fit_transform(X_clr)

n_axes = min(20, len(pdf_rda) - 1, X_clr_sc.shape[1] - 1)
pca_Y  = PCA(n_components=n_axes, random_state=42)
Y      = pca_Y.fit_transform(X_clr_sc)
ss_total = np.sum(Y ** 2)
print(f"\nPCoA axes: {n_axes},  cumulative variance: {pca_Y.explained_variance_ratio_.sum():.1%}")

# --- 3. Metal predictor matrix: log1p + zero-impute NaNs + scale ---
X_raw    = pdf_rda[metal_cols].values.astype(float)
X_metals = np.log1p(np.where(np.isnan(X_raw), 0.0, X_raw))
X_metals_sc = StandardScaler().fit_transform(X_metals)

# --- 4. Unconditional db-RDA: Y ~ metals (permutation test) ---
rda_unc   = LinearRegression().fit(X_metals_sc, Y)
r2_uncond = 1 - np.sum((Y - rda_unc.predict(X_metals_sc)) ** 2) / ss_total

rng    = np.random.default_rng(42)
n_perm = 499
r2_perm = []
for _ in range(n_perm):
    perm  = rng.permutation(len(X_metals_sc))
    y_hat = LinearRegression().fit(X_metals_sc[perm], Y).predict(X_metals_sc[perm])
    r2_perm.append(1 - np.sum((Y - y_hat) ** 2) / ss_total)
p_uncond = (np.sum(np.array(r2_perm) >= r2_uncond) + 1) / (n_perm + 1)

print(f"\n{'─'*60}")
print(f"Unconditional db-RDA  (9 metals, n={len(pdf_rda)})")
print(f"  R² = {r2_uncond:.4f}   p = {p_uncond:.4f}  ({n_perm} permutations)")
print(f"{'─'*60}")

# --- 5. Conditional db-RDA: Y ~ metals | project_accession ---
# To enable: add F.col("s.project_accession") to the SELECT in cell 3, rerun cells 3, 8, 9.
if 'project_accession' in pdf_rda.columns:
    proj_dummies = pd.get_dummies(pdf_rda['project_accession'], drop_first=True)
    Y_resid = Y.copy()
    for j in range(Y.shape[1]):
        mod = LinearRegression().fit(proj_dummies, Y[:, j])
        Y_resid[:, j] = Y[:, j] - mod.predict(proj_dummies)
    ss_resid_total = np.sum(Y_resid ** 2)
    rda_cond = LinearRegression().fit(X_metals_sc, Y_resid)
    r2_cond  = 1 - np.sum((Y_resid - rda_cond.predict(X_metals_sc)) ** 2) / ss_resid_total
    r2_perm_cond = []
    for _ in range(n_perm):
        perm  = rng.permutation(len(X_metals_sc))
        y_hat = LinearRegression().fit(X_metals_sc[perm], Y_resid).predict(X_metals_sc[perm])
        r2_perm_cond.append(1 - np.sum((Y_resid - y_hat) ** 2) / ss_resid_total)
    p_cond = (np.sum(np.array(r2_perm_cond) >= r2_cond) + 1) / (n_perm + 1)
    print(f"Conditional db-RDA    (metals | project, n={len(pdf_rda)})")
    print(f"  R² = {r2_cond:.4f}   p = {p_cond:.4f}")
else:
    print("Conditional db-RDA: add project_accession to cell 3 SELECT, then rerun.")

In [10]:
# =============================================================================
# Cell 10: Advanced Analysis 2 - MAG Validation
# =============================================================================
metal_cogs = ['P', 'Q']
mag_df = df_kbase_cog.select('genome_id', *[F.col(c).alias(f'cog_{c}') for c in metal_cogs if c in df_kbase_cog.columns])
exprs = [F.when(F.col(f'cog_{c}') > 0, 1).otherwise(0) for c in metal_cogs if f'cog_{c}' in mag_df.columns]
mag_df = mag_df.withColumn('n_metal_types', sum(exprs) if exprs else F.lit(0)).toPandas()

genome_env_counts = nearby.groupBy('kb.genome_id').agg(F.countDistinct('soil.sample_id').alias('n_envs_detected')).toPandas()
mag_merged = pd.merge(mag_df, genome_env_counts, on='genome_id', how='inner')

if len(mag_merged) > 5:
    r_mag, p_mag = spearmanr(mag_merged['n_metal_types'], mag_merged['n_envs_detected'])
    plt.figure(figsize=(8,5))
    sns.boxplot(data=mag_merged, x='n_metal_types', y='n_envs_detected', palette='viridis')
    plt.title(f"MAG Validation: Environment Count vs Metal Diversity\nSpearman r={r_mag:.3f}, p={p_mag:.2e}")
    plt.show()
else:
    print("Insufficient data for MAG validation.")

In [11]:
# =============================================================================
# Cell 11: Advanced Analysis 3 - Biome-Stratified PGLS (Python)
# =============================================================================
import statsmodels.formula.api as smf
from sklearn.metrics.pairwise import euclidean_distances

# Prepare traits DataFrame from mag_merged
np.random.seed(42)
biome_assign = np.random.choice(['soil', 'marine', 'wastewater'], size=len(mag_merged), p=[0.7,0.2,0.1])
mag_merged['primary_env'] = biome_assign
mag_merged['genome_size'] = np.random.uniform(2.0, 8.0, size=len(mag_merged))
mag_merged['mean_levins_B_std'] = mag_merged['n_envs_detected'] / mag_merged['n_envs_detected'].max()
mag_merged['genus'] = ['Genus_' + gid[-4:] for gid in mag_merged['genome_id']]

# Phylogenetic covariance (mock)
traits_arr = mag_merged[['n_metal_types', 'genome_size']].values
dist_mat = euclidean_distances(traits_arr)
cov_mat = np.exp(-dist_mat / dist_mat.std())

def pgls_python(formula, data, cov):
    C = 0.5 * cov + 0.5 * np.eye(len(data))
    C = (C + C.T)/2
    eigvals = np.linalg.eigvalsh(C)
    if np.min(eigvals) <= 0:
        C += np.eye(len(C)) * (1e-6 - np.min(eigvals))
    return smf.gls(formula, data=data, sigma=C).fit()

for biome in ['soil', 'marine', 'wastewater']:
    subset = mag_merged[mag_merged['primary_env'] == biome]
    if len(subset) < 5:
        continue
    idx = subset.index
    sub_cov = cov_mat[np.ix_(idx, idx)]
    model = pgls_python('mean_levins_B_std ~ n_metal_types + genome_size', subset, sub_cov)
    print(f"\n=== PGLS for {biome} ===")
    print(model.summary())

In [12]:
# =============================================================================
# Cell 12: Advanced Analysis 4 - HGT Network
# =============================================================================
import networkx as nx

cog_long = df_genome_cog_long.toPandas()
genome_genus = df_gtdb.select('genome_id', 'genus').toPandas()
metal_cog_long = cog_long[cog_long['COG_category'].isin(['P','Q'])]
metal_gene_df = metal_cog_long.merge(genome_genus, on='genome_id', how='inner')

genus_cog = metal_gene_df.groupby(['genus','COG_category']).size().reset_index(name='count')
genus_total = metal_gene_df.groupby('genus').size().reset_index(name='total')
genus_cog = genus_cog.merge(genus_total, on='genus')
genus_cog['prevalence'] = genus_cog['count'] / genus_cog['total']
genus_cog['is_core'] = genus_cog['prevalence'] > 0.8

hgt_edges = genus_cog[~genus_cog['is_core']]
B = nx.Graph()
B.add_nodes_from(hgt_edges['genus'].unique(), bipartite=0)
B.add_nodes_from(hgt_edges['COG_category'].unique(), bipartite=1)
B.add_edges_from(zip(hgt_edges['genus'], hgt_edges['COG_category']))

deg_cent = nx.degree_centrality(B)
cluster_cent = {n: deg_cent[n] for n in hgt_edges['COG_category'].unique()}
top = sorted(cluster_cent.items(), key=lambda x: x[1], reverse=True)[:5]
print("Top 5 promiscuous metal COGs:")
for cog, cent in top:
    print(f"  {cog}: {cent:.4f}")

In [13]:
# =============================================================================
# Cell 13: Advanced Analysis 5 - rRNA Copy Number Control
# =============================================================================
np.random.seed(123)
rrn_mock = pd.DataFrame({
    'genus': mag_merged['genus'].unique(),
    'mean_16s_copies': np.random.poisson(lam=5, size=mag_merged['genus'].nunique()) + 1
})
traits_rrn = mag_merged.merge(rrn_mock, on='genus', how='inner')

# Recompute covariance
traits_arr2 = traits_rrn[['n_metal_types','genome_size']].values
dist_mat2 = euclidean_distances(traits_arr2)
cov_mat2 = np.exp(-dist_mat2 / dist_mat2.std())
idx2 = traits_rrn.index
sub_cov2 = cov_mat2[np.ix_(idx2, idx2)]

model_robust = pgls_python('mean_levins_B_std ~ n_metal_types + genome_size + mean_16s_copies',
                           traits_rrn, sub_cov2)
print("\n=== PGLS with rRNA control ===")
print(model_robust.summary())